In [ ]:
import os

# Your Kaggle API token, so you don't have to log in by hand every time.
# Remove or change this before sharing your code.
os.environ["KAGGLE_API_TOKEN"] = "removed"

import kagglehub


In [3]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('tabular-time-series-trading-signal-classification')

print("Path to competition files:", path)

Path to competition files: /Users/mikkelengelbaek/.cache/kagglehub/competitions/tabular-time-series-trading-signal-classification


In [4]:
from pathlib import Path
import shutil
import numpy as np
import pandas as pd
import kagglehub

path = kagglehub.competition_download(
    "tabular-time-series-trading-signal-classification",
    force_download=True,  # get the new additional-features files too
)

# Collect every CSV from the download (wherever it ended up) into a local "data" folder.
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)
for csv_file in Path(path).rglob("*.csv"):
    shutil.copy(csv_file, data_dir / csv_file.name)
print("Files found:", sorted(p.name for p in data_dir.glob("*.csv")))

train = pd.read_csv(data_dir / "btc_train.csv")
test = pd.read_csv(data_dir / "btc_test.csv")
sample_submission = pd.read_csv(data_dir / "sample_submission.csv")

print(train.head())
print(test.head())
print(sample_submission.head())


100%|██████████| 406k/406k [00:01<00:00, 383kB/s]

Extracting files...


Files found: ['btc_test.csv', 'btc_train.csv', 'sample_submission.csv', 'test_additional_features.csv', 'train_additional_features.csv']
   Id             datetime     prices  Signal
0   0  2025-09-29 08:00:00  112099.75       0
1   1  2025-09-29 09:00:00  112110.90       0
2   2  2025-09-29 10:00:00  112090.18       0
3   3  2025-09-29 11:00:00  112100.00       0
4   4  2025-09-29 12:00:00  112022.16       1
     Id             datetime    prices
0  6132  2026-06-11 20:00:00  63393.67
1  6133  2026-06-11 21:00:00  63519.99
2  6134  2026-06-11 22:00:00  63500.01
3  6135  2026-06-11 23:00:00  63625.99
4  6136  2026-06-12 00:00:00  63648.80
     Id  Signal
0  6132       0
1  6133       0
2  6134       0
3  6135       0
4  6136       0


In [5]:
train["datetime"] = pd.to_datetime(train["datetime"])

# Convert datetime into numeric time features.
train["time_index"] = (train["datetime"] - train["datetime"].min()).dt.total_seconds()
train["hour"] = train["datetime"].dt.hour
train["day_of_week"] = train["datetime"].dt.dayofweek
train["month"] = train["datetime"].dt.month

# Preserve continuity between, for example, 23:00 and 00:00.
for column, period in [("hour", 24), ("day_of_week", 7), ("month", 12)]:
    train[f"{column}_sin"] = np.sin(2 * np.pi * train[column] / period)
    train[f"{column}_cos"] = np.cos(2 * np.pi * train[column] / period)

# Create useful price-change features.
train["price_lag_1"] = train["prices"].shift(1)
train["price_change_1"] = train["prices"] - train["price_lag_1"]
train["price_return_1"] = train["price_change_1"] / train["price_lag_1"]
train["price_lag_24"] = train["prices"].shift(24)
train["price_change_24"] = train["prices"] - train["price_lag_24"]

# Remove the identifier and raw datetime before modeling.
feature_columns = [
    "time_index", "hour", "day_of_week", "month",
    "hour_sin", "hour_cos", "day_of_week_sin", "day_of_week_cos",
    "month_sin", "month_cos", "price_lag_1", "price_change_1",
    "price_return_1", "price_lag_24", "price_change_24", "prices"
]

features = train[feature_columns].copy()
features["Signal"] = train["Signal"]
features = features.dropna().reset_index(drop=True)

print(features.head())

   time_index  hour  day_of_week  month      hour_sin  hour_cos  \
0     86400.0     8            1      9  8.660254e-01 -0.500000   
1     90000.0     9            1      9  7.071068e-01 -0.707107   
2     93600.0    10            1      9  5.000000e-01 -0.866025   
3     97200.0    11            1      9  2.588190e-01 -0.965926   
4    100800.0    12            1      9  1.224647e-16 -1.000000   

   day_of_week_sin  day_of_week_cos  month_sin     month_cos  price_lag_1  \
0         0.781831          0.62349       -1.0 -1.836970e-16    113699.23   
1         0.781831          0.62349       -1.0 -1.836970e-16    113451.89   
2         0.781831          0.62349       -1.0 -1.836970e-16    112781.97   
3         0.781831          0.62349       -1.0 -1.836970e-16    112923.49   
4         0.781831          0.62349       -1.0 -1.836970e-16    113055.01   

   price_change_1  price_return_1  price_lag_24  price_change_24     prices  \
0         -247.34       -0.002175     112099.75        

In [6]:
import io
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier

# =====================================================================================
# competition7: competition6 + public Binance 1-minute candles, used for ONE thing: a sharper
# estimate of how volatile the market is right now, for the two simulated-paths members.
# Six "opinions" about each dip, averaged:
#
#   1. LR trimmed + volume      logistic regression: volatility, time, last move, volume
#   2. LR volatility            logistic regression: volatility and time of day
#   3. Paths (day)              simulated price paths, volatility pattern by hour of day      <- now sized
#   4. Paths (week)             simulated price paths, volatility pattern by hour of the week <- with minutes
#   5. XGB dips + volume        small XGBoost trained only on dip candidates
#   6. XGB original (returns)   your original XGBoost on all hours, using price CHANGES and time
#                               of day only (raw price levels / calendar dropped), non-dips set to 0
#
# Other minute-level ideas (moves in the last 5/15/30 minutes, selling slowing down, when the
# hour's low happened, buyer/seller balance in the final minutes) were tested on real data and did
# not help, so they are not included.
# Every feature only looks backwards: an hourly row only ever sees the minutes INSIDE the hour that
# ends at its price, and earlier. Hours that can't be a signal get 0.
# External data used (remember to declare it on the competition forum):
#   - Binance BTCUSDT 1h klines (data.binance.vision), only hours BEFORE the competition period
#   - Binance BTCUSDT 1m klines (data.binance.vision), Jan 2022 up to the end of the test period
# =====================================================================================

# -------------------------------
# 1. Competition data
# -------------------------------
train = pd.read_csv(data_dir / "btc_train.csv")
test = pd.read_csv(data_dir / "btc_test.csv")
for frame in (train, test):
    frame["datetime"] = pd.to_datetime(frame["datetime"])
train = train.sort_values("datetime").reset_index(drop=True)
test = test.sort_values("datetime").reset_index(drop=True)
competition = pd.concat([train[["datetime", "prices"]], test[["datetime", "prices"]]], ignore_index=True)

ACTIVITY_COLUMNS = ["volume", "quote_asset_volume", "number_of_trades",
                    "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume"]
if not (data_dir / "train_additional_features.csv").exists():
    raise FileNotFoundError("New data files not found. Re-download with "
                            "kagglehub.competition_download(..., force_download=True).")
extra = pd.concat([pd.read_csv(data_dir / "train_additional_features.csv"),
                   pd.read_csv(data_dir / "test_additional_features.csv")], ignore_index=True)
extra["datetime"] = pd.to_datetime(extra["datetime"])
extra = extra.sort_values("datetime").reset_index(drop=True)
assert (extra["datetime"].to_numpy() == competition["datetime"].to_numpy()).all(), "Rows don't line up"
assert (extra["Id"].to_numpy() == np.r_[train["Id"], test["Id"]]).all(), "Ids don't line up"

# Timing check (as in competition4): each activity row must cover the hour that ENDS at that
# row's price. The hour's average traded price moves with the price change INTO the row if so.
CHECK_HOURS = 2000
competition_returns = np.log(competition["prices"]).diff()
gap = np.log(extra["quote_asset_volume"] / extra["volume"] / competition["prices"])
corr_into = gap.iloc[1:CHECK_HOURS].corr(competition_returns.iloc[1:CHECK_HOURS])
corr_out_of = gap.iloc[1:CHECK_HOURS].corr(competition_returns.shift(-1).iloc[1:CHECK_HOURS])
covers_hour_after = abs(corr_out_of) > abs(corr_into) or max(abs(corr_into), abs(corr_out_of)) < 0.3
print(f"Timing check: link with price change into the row {corr_into:+.2f}, out of the row {corr_out_of:+.2f}")
if covers_hour_after:
    competition[ACTIVITY_COLUMNS] = extra[ACTIVITY_COLUMNS].shift(1).to_numpy()
    print("-> Activity rows cover the hour AFTER each price. Shifted down one row to avoid future data.")
else:
    competition[ACTIVITY_COLUMNS] = extra[ACTIVITY_COLUMNS].to_numpy()
    print("-> Activity rows cover the hour that ends at each price. Used as they are.")


# -------------------------------
# 2. Older public data: Binance BTCUSDT hourly candles, Jan 2022 - Sep 2025
# -------------------------------
# The competition's prices and volumes are exactly Binance BTCUSDT hourly data (the price
# stamped 08:00 is the close of the candle that opens at 08:00), so older Binance candles
# extend the same series backwards. Downloaded once, then reused from the data folder.
BINANCE_FILE = data_dir / "binance_btcusdt_1h_2022_2025.csv"
BINANCE_COLUMNS = ["open_time", "open", "high", "low", "close", "volume", "close_time",
                   "quote_asset_volume", "number_of_trades", "taker_buy_base_asset_volume",
                   "taker_buy_quote_asset_volume", "ignore"]
if not BINANCE_FILE.exists() and Path("binance_btcusdt_1h_2022_2025.csv").exists():
    BINANCE_FILE = Path("binance_btcusdt_1h_2022_2025.csv")  # file made earlier with the download script
if not BINANCE_FILE.exists():
    print("Downloading Binance BTCUSDT 1h candles 2022-01 .. 2025-09 (about 45 files)...")
    frames = []
    for month in pd.period_range("2022-01", "2025-09", freq="M"):
        url = f"https://data.binance.vision/data/spot/monthly/klines/BTCUSDT/1h/BTCUSDT-1h-{month}.zip"
        with urllib.request.urlopen(url) as response:
            archive = zipfile.ZipFile(io.BytesIO(response.read()))
            raw = pd.read_csv(archive.open(archive.namelist()[0]), header=None)
        if not str(raw.iloc[0, 0]).isdigit():  # skip a header row if there is one
            raw = raw.iloc[1:]
        frames.append(raw)
    binance = pd.concat(frames, ignore_index=True)
    binance.columns = BINANCE_COLUMNS
    binance = binance.apply(pd.to_numeric)
    times = binance["open_time"].astype("int64")
    times = times.where(times < 1e14, times // 1000)  # newer files use microseconds
    binance["open_time"] = pd.to_datetime(times, unit="ms")
    binance.drop(columns="ignore").to_csv(data_dir / "binance_btcusdt_1h_2022_2025.csv", index=False)
    BINANCE_FILE = data_dir / "binance_btcusdt_1h_2022_2025.csv"
binance = pd.read_csv(BINANCE_FILE)
binance["datetime"] = pd.to_datetime(binance["open_time"], format="ISO8601")
binance = binance.drop_duplicates("datetime").sort_values("datetime")

# Sanity check on the hours where both sources exist.
overlap = train.merge(binance, on="datetime")
print(f"Binance close == competition price on {np.mean(np.abs(overlap['prices'] - overlap['close']) < 0.011):.0%} "
      f"of {len(overlap)} overlapping hours")

# Keep ONLY hours before the competition's training period starts.
competition_start = train["datetime"].min()
older = binance[binance["datetime"] < competition_start]
all_hours = pd.DataFrame({"datetime": pd.date_range(older["datetime"].min(),
                                                    competition_start - pd.Timedelta(hours=1), freq="h")})
older = all_hours.merge(older, on="datetime", how="left").ffill()  # fills the odd missing hour
older = older.rename(columns={"close": "prices"})
no_trading = older["volume"] <= 0                                  # exchange outages
older.loc[no_trading, ACTIVITY_COLUMNS] = np.nan
older[ACTIVITY_COLUMNS] = older[ACTIVITY_COLUMNS].ffill()
print(f"Older history: {older['datetime'].min()} -> {older['datetime'].max()} ({len(older)} hours)")


# -------------------------------
# 3. One timeline: older history + competition train + competition test
# -------------------------------
timeline = pd.concat([older[["datetime", "prices"] + ACTIVITY_COLUMNS],
                      competition[["datetime", "prices"] + ACTIVITY_COLUMNS]], ignore_index=True)
assert (timeline["datetime"].diff().dropna() == pd.Timedelta(hours=1)).all(), "Timeline has gaps"
n_older, n_train, n_test = len(older), len(train), len(test)
train_end = n_older + n_train            # rows [train_end:] are the test period
prices = timeline["prices"].astype(float)
log_price = np.log(prices)
return_1h = log_price.diff()
hour = timeline["datetime"].dt.hour
day_of_week = timeline["datetime"].dt.dayofweek
activity = timeline[ACTIVITY_COLUMNS].astype(float)


# -------------------------------
# 3b. Public Binance BTCUSDT 1-minute candles: how much the price wiggled inside each hour
# -------------------------------
# Row t's price is the close of the hour from t to t+60min. Row t only ever uses the 60 minutes
# that open at t, t+1min, ..., t+59min (the last of these closes exactly when row t's price is
# set) plus earlier hours. The file is cut strictly at the end of the test period.
TEST_END = test["datetime"].max() + pd.Timedelta(hours=1)          # when the last test price is set
TEST_END_MS = int((TEST_END - pd.Timestamp("1970-01-01")) // pd.Timedelta(milliseconds=1))
MINUTE_COLUMNS = ["open_time", "open", "high", "low", "close", "volume", "number_of_trades",
                  "taker_buy_base_asset_volume"]
MINUTE_FILE = data_dir / "binance_btcusdt_1m_2022_2026.csv.gz"
for earlier_copy in [Path("binance_btcusdt_1m_2022_2026.csv.gz"), Path("binance_btcusdt_1m_2022_2026_csv.gz"),
                     data_dir / "binance_btcusdt_1m_2022_2026_csv.gz"]:
    if not MINUTE_FILE.exists() and earlier_copy.exists():
        MINUTE_FILE = earlier_copy                                   # file made earlier with the download script


def read_kline_zip(url):
    with urllib.request.urlopen(url) as response:
        archive = zipfile.ZipFile(io.BytesIO(response.read()))
        raw = pd.read_csv(archive.open(archive.namelist()[0]), header=None, usecols=[0, 1, 2, 3, 4, 5, 8, 9])
    if not str(raw.iloc[0, 0]).isdigit():  # skip a header row if there is one
        raw = raw.iloc[1:]
    return raw


if not MINUTE_FILE.exists():
    last_minute = TEST_END - pd.Timedelta(minutes=1)
    months = pd.period_range("2022-01", last_minute.to_period("M"), freq="M")
    print(f"Downloading Binance BTCUSDT 1m candles {months[0]} .. {months[-1]} ({len(months)} files, "
          f"~150 MB, a few minutes)...")
    BASE_URL = "https://data.binance.vision/data/spot"
    frames = []
    for month in months:
        try:
            frames.append(read_kline_zip(f"{BASE_URL}/monthly/klines/BTCUSDT/1m/BTCUSDT-1m-{month}.zip"))
        except urllib.error.HTTPError:  # month not published as one file yet -> use its daily files
            for day in pd.date_range(month.start_time.normalize(), min(month.end_time, last_minute).normalize(), freq="D"):
                frames.append(read_kline_zip(f"{BASE_URL}/daily/klines/BTCUSDT/1m/BTCUSDT-1m-{day:%Y-%m-%d}.zip"))
    minutes = pd.concat(frames, ignore_index=True)
    minutes.columns = MINUTE_COLUMNS
    minutes = minutes.apply(pd.to_numeric)
    times = minutes["open_time"].astype("int64")
    minutes["open_time"] = times.where(times < 1e14, times // 1000)  # newer files use microseconds
    minutes = minutes[minutes["open_time"] < TEST_END_MS]
    minutes.to_csv(data_dir / "binance_btcusdt_1m_2022_2026.csv.gz", index=False, compression="gzip")
    MINUTE_FILE = data_dir / "binance_btcusdt_1m_2022_2026.csv.gz"
minutes = pd.read_csv(MINUTE_FILE, compression="gzip")
if not pd.api.types.is_numeric_dtype(minutes["open_time"]):         # a file with readable timestamps
    minutes["open_time"] = (pd.to_datetime(minutes["open_time"]) - pd.Timestamp("1970-01-01")) // pd.Timedelta(milliseconds=1)
minutes = minutes[minutes["open_time"] < TEST_END_MS]                 # nothing after the test period, ever
minutes = minutes.drop_duplicates("open_time").sort_values("open_time")
minute_time = pd.to_datetime(minutes["open_time"], unit="ms").astype("datetime64[ns]")
print(f"Minute candles: {minute_time.min()} -> {minute_time.max()} ({len(minutes):,} minutes)")

# One row per timeline hour, one column per minute of that hour (minute 0 opens at the row's datetime).
n_hours = len(timeline)
grid_times = pd.date_range(pd.Timestamp(timeline["datetime"].iloc[0]).as_unit("ns"), periods=n_hours * 60, freq="min")
grid = minutes.set_index(minute_time).reindex(grid_times)
last_minute_opens = grid_times.to_numpy().reshape(n_hours, 60)[:, -1]
assert (last_minute_opens == (timeline["datetime"] + pd.Timedelta(minutes=59)).to_numpy()).all(), "Minute grid misaligned"
assert grid_times[-1] < TEST_END, "Minute grid reaches past the test period"
recorded = grid["close"].notna().to_numpy().reshape(n_hours, 60)
close_grid = grid["close"].ffill().to_numpy().reshape(n_hours, 60)  # an unrecorded minute keeps the last price

# Check that the minutes line up EXACTLY with the competition's hourly prices: the last minute of
# each hour must close at that hour's competition price, and the minute volumes must add up.
competition_rows = np.arange(n_older, n_hours)
same_close = np.abs(close_grid[competition_rows, -1] - prices.to_numpy()[competition_rows]) < 0.005
print(f"Last minute of the hour closes at the competition price: {same_close[:n_train].mean():.2%} of training hours, "
      f"{same_close[n_train:].mean():.2%} of test hours")
for row in competition_rows[~same_close]:
    # (the competition's very last test price is a snapshot taken during that hour, so it can differ)
    print(f"   differs at {timeline['datetime'][row]}: competition {prices[row]:.2f}, last minute {close_grid[row, -1]:.2f}")
if same_close.mean() < 0.999:
    raise ValueError("The minute data does not line up with the competition prices - check the minute file.")
minute_volume = grid["volume"].fillna(0).to_numpy().reshape(n_hours, 60).sum(axis=1)
same_volume = np.isclose(minute_volume[competition_rows], activity["volume"].to_numpy()[competition_rows], rtol=1e-6)
older_close = np.abs(close_grid[:n_older, -1] - prices.to_numpy()[:n_older]) < 0.005
print(f"Minute volumes add up to the competition's hourly volume: {same_volume.mean():.2%} of hours | "
      f"last minute == older Binance hourly close: {older_close.mean():.2%} of older hours")

# Realized variance: the sum of squared one-minute moves inside the hour (an hour where only part of
# the minutes were recorded is scaled up; hours with fewer than 30 recorded minutes are left out).
log_close_grid = np.log(close_grid)
previous_close = np.r_[np.nan, log_close_grid[:-1, -1]]
minute_moves = np.diff(np.c_[previous_close, log_close_grid], axis=1)   # 60 one-minute moves per hour
n_recorded = recorded.sum(axis=1)
realized_variance = np.nansum(minute_moves ** 2, axis=1) * 60 / np.maximum(n_recorded, 1)
realized_variance[n_recorded < 30] = np.nan
print(f"Hours without enough minutes (left out): {int((n_recorded < 30).sum())}")


# -------------------------------
# 4. The "left half" of the signal rule, features, dip candidates (as competition4)
# -------------------------------
def left_side_of_dip(price_values):
    """dip_depth: how far the price is below the highest price since the last time the price
    was lower than now (log scale, 0.01 ~ 1%); hours_since_high: how long ago that high was."""
    price_values = np.asarray(price_values, dtype=float)
    dip_depth = np.zeros(len(price_values))
    hours_since_high = np.zeros(len(price_values))
    stack = []  # (price, highest price in its stretch, index of that high)
    for t, price in enumerate(price_values):
        high, high_index = price, t
        while stack and stack[-1][0] >= price:
            _, earlier_high, earlier_index = stack.pop()
            if earlier_high > high:
                high, high_index = earlier_high, earlier_index
        stack.append((price, high, high_index))
        dip_depth[t] = np.log(high / price)
        hours_since_high[t] = t - high_index
    return dip_depth, hours_since_high


features = pd.DataFrame(index=timeline.index)
features["hour_sin"] = np.sin(2 * np.pi * hour / 24)
features["hour_cos"] = np.cos(2 * np.pi * hour / 24)
features["day_of_week_sin"] = np.sin(2 * np.pi * day_of_week / 7)
features["day_of_week_cos"] = np.cos(2 * np.pi * day_of_week / 7)
features["is_weekend"] = (day_of_week >= 5).astype(float)
for lag in [1, 2, 3, 6, 24]:
    features[f"price_return_{lag}"] = prices / prices.shift(lag) - 1
for k in [1, 2, 3, 4]:
    features[f"hourly_return_lag{k}"] = return_1h.shift(k)
for window in [6, 24, 72, 168]:
    features[f"log_vol_{window}"] = np.log(return_1h.rolling(window).std() + 1e-9)
volatility_24 = np.exp(features["log_vol_24"])
dip_depth, hours_since_high = left_side_of_dip(prices)
features["dip_depth"] = dip_depth
features["log_hours_since_high"] = np.log1p(hours_since_high)
features["dip_depth_vs_vol"] = dip_depth / (volatility_24 + 1e-9)
features["return_1_vs_vol"] = return_1h / (volatility_24 + 1e-9)
features["vol_ratio_6_72"] = features["log_vol_6"] - features["log_vol_72"]
for window in [24, 72, 168]:
    features[f"drawdown_{window}"] = log_price - np.log(prices.rolling(window).max())
for window in [24, 72]:
    low, high = prices.rolling(window).min(), prices.rolling(window).max()
    features[f"range_position_{window}"] = (prices - low) / (high - low + 1e-9)
features["dist_ema_12"] = log_price - np.log(prices.ewm(span=12, adjust=False).mean())

volume = activity["volume"]
trades = activity["number_of_trades"]
buyer_volume = activity["taker_buy_base_asset_volume"]
hour_average_price = activity["quote_asset_volume"] / volume
features["log_volume_rel_24"] = np.log(volume / volume.shift(1).rolling(24).mean())
features["log_volume_rel_168"] = np.log(volume / volume.shift(1).rolling(168).mean())
features["log_volume_6h_rel_168"] = np.log(volume.rolling(6).mean() / volume.shift(6).rolling(168).mean())
features["log_trades_rel_168"] = np.log(trades / trades.shift(1).rolling(168).mean())
trade_size = volume / trades
features["log_trade_size_rel_168"] = np.log(trade_size / trade_size.shift(1).rolling(168).mean())
buyer_share = buyer_volume / volume
features["buyer_share"] = buyer_share
features["buyer_share_3h_vs_week"] = (
    buyer_volume.rolling(3).sum() / volume.rolling(3).sum() - buyer_share.shift(3).rolling(168).mean()
)
features["close_vs_hour_average"] = np.log(prices / hour_average_price) / (volatility_24 + 1e-9)
features = features.replace([np.inf, -np.inf], np.nan)

features["dip_candidate"] = ((features["price_return_1"] <= 0) & (features["dip_depth"] >= 0.0099)).astype(int)
is_candidate = features["dip_candidate"].to_numpy() == 1

LR_TRIM_COLUMNS = ["log_vol_6", "log_vol_24", "log_vol_72", "hour_sin", "hour_cos", "is_weekend",
                   "return_1_vs_vol", "log_hours_since_high"]
LR_TRIM_VOLUME_COLUMNS = LR_TRIM_COLUMNS + ["log_volume_rel_168", "log_volume_6h_rel_168",
                                            "buyer_share_3h_vs_week", "close_vs_hour_average"]
LR_VOL_COLUMNS = ["log_vol_6", "log_vol_24", "log_vol_72", "hour_sin", "hour_cos"]
XGB_DIP_VOLUME_COLUMNS = [
    "log_vol_6", "log_vol_24", "log_vol_72", "log_vol_168", "vol_ratio_6_72",
    "hour_sin", "hour_cos", "day_of_week_sin", "day_of_week_cos", "is_weekend",
    "return_1_vs_vol", "dip_depth_vs_vol", "log_hours_since_high", "dip_depth",
    "price_return_1", "price_return_2", "price_return_3", "price_return_6", "price_return_24",
    "hourly_return_lag1", "hourly_return_lag2", "hourly_return_lag3", "hourly_return_lag4",
    "drawdown_24", "drawdown_72", "drawdown_168", "range_position_24", "range_position_72",
    "dist_ema_12", "log_volume_rel_24", "log_volume_rel_168", "log_volume_6h_rel_168", "log_trades_rel_168",
    "log_trade_size_rel_168", "buyer_share", "buyer_share_3h_vs_week", "close_vs_hour_average",
]


# -------------------------------
# 5. Simulated future price paths ("bounce probability")
# -------------------------------
# For every dip: simulate 2,000 possible next 48 hours and count how often the price climbs
# 1% before it drops below the dip. Each simulated hour moves by a random amount sized by
# recent volatility and by how busy that hour usually is. The "how busy" pattern is learned
# from the long history, with recent months counting more (half-life ~6 months).
# NEW in competition7: "recent volatility" comes from the minute-level wiggles of the last ~day
# (realized variance) instead of the squared hourly moves - 60 measurements per hour instead of 1,
# so it reacts faster and is less noisy. A slowly updated ratio (past data only) puts it on the
# scale of hourly moves.
UP_TARGET = np.log(1.01)
N_PATHS, HORIZON = 2000, 48
PATTERN_HALF_LIFE_HOURS = 6 * 730
returns = return_1h.to_numpy()


def volatility_pattern(returns, buckets, n_buckets, prior=None, shrink=24.0, half_life=PATTERN_HALF_LIFE_HOURS):
    """For every hour t: how volatile each bucket (hour of day / hour of week) has been so far,
    relative to average (1.0 = average), weighting recent data more. Data up to hour t only."""
    squared = np.nan_to_num(returns ** 2)
    one_hot = np.zeros((len(returns), n_buckets))
    one_hot[np.arange(len(returns)), buckets] = 1.0
    one_hot[np.isnan(returns)] = 0.0
    alpha = 1 - 0.5 ** (1.0 / half_life)
    effective_n = (1 - (1 - alpha) ** np.arange(1, len(returns) + 1)) / alpha
    sums = pd.DataFrame(one_hot * squared[:, None]).ewm(alpha=alpha).mean().to_numpy() * effective_n[:, None]
    counts = pd.DataFrame(one_hot).ewm(alpha=alpha).mean().to_numpy() * effective_n[:, None]
    overall = sums.sum(axis=1) / np.maximum(counts.sum(axis=1), 1e-12)
    target = overall[:, None] if prior is None else prior
    per_bucket = (sums + shrink * target) / (counts + shrink)
    factors = np.sqrt(per_bucket / np.maximum(overall[:, None], 1e-18))
    factors[overall <= 0] = 1.0
    return factors, per_bucket


def bounce_probabilities(returns, buckets, factors, future_buckets, is_candidate, span=24, seed=0,
                         minute_variance=None):
    own_factor = factors[np.arange(len(returns)), buckets]
    hourly_based = pd.Series((returns / own_factor) ** 2)
    if minute_variance is None:   # competition6: recent squared hourly moves
        recent_variance = hourly_based.ewm(span=span, ignore_na=True).mean().to_numpy()
    else:                         # competition7: recent minute-level realized variance, on the hourly scale
        minute_based = pd.Series(minute_variance / own_factor ** 2)
        to_hourly_scale = (hourly_based.ewm(halflife=PATTERN_HALF_LIFE_HOURS, ignore_na=True).mean()
                           / minute_based.ewm(halflife=PATTERN_HALF_LIFE_HOURS, ignore_na=True).mean()).to_numpy()
        recent_variance = minute_based.ewm(span=span, ignore_na=True).mean().to_numpy() * to_hourly_scale
    rng = np.random.default_rng(seed)
    shocks = rng.standard_t(4, size=(N_PATHS, HORIZON)) / np.sqrt(2.0)  # fat tails, variance 1
    probabilities = np.zeros(len(returns))
    for t in np.flatnonzero(is_candidate):
        if not np.isfinite(recent_variance[t]):
            continue
        hourly_size = np.sqrt(recent_variance[t]) * factors[t, future_buckets[t]]
        paths = np.cumsum(shocks * hourly_size, axis=1)  # log price relative to the dip
        hit_up, hit_down = paths >= UP_TARGET, paths < 0
        first_up = np.where(hit_up.any(axis=1), hit_up.argmax(axis=1), HORIZON)
        first_down = np.where(hit_down.any(axis=1), hit_down.argmax(axis=1), HORIZON)
        probabilities[t] = np.mean(first_up < first_down)
    return probabilities


hour_values = hour.to_numpy()
hour_of_week = (day_of_week * 24 + hour).to_numpy()
future_hour_of_week = (hour_of_week[:, None] + np.arange(1, HORIZON + 1)[None, :]) % 168
day_factors, day_variance = volatility_pattern(returns, hour_values, 24)
week_factors, _ = volatility_pattern(returns, hour_of_week, 168, prior=day_variance[:, np.arange(168) % 24])
features["paths_day"] = bounce_probabilities(returns, hour_values, day_factors, future_hour_of_week % 24,
                                             is_candidate, minute_variance=realized_variance)
features["paths_week"] = bounce_probabilities(returns, hour_of_week, week_factors, future_hour_of_week,
                                              is_candidate, minute_variance=realized_variance)
# competition6's versions (hourly moves only), kept only for the comparison in the validation
features["paths_day_hourly"] = bounce_probabilities(returns, hour_values, day_factors, future_hour_of_week % 24,
                                                    is_candidate)
features["paths_week_hourly"] = bounce_probabilities(returns, hour_of_week, week_factors, future_hour_of_week,
                                                     is_candidate)


# -------------------------------
# 6. Features for the original XGBoost: price changes + time of day/week only
# -------------------------------
# (Raw price levels, time index, month and day of year were dropped: with 4.5 years of data
#  they describe old market regimes rather than anything that carries over to the test period.)
original = pd.DataFrame(index=timeline.index)
original["hour"] = hour
original["day_of_week"] = day_of_week
for column, period in [("hour", 24), ("day_of_week", 7)]:
    original[f"{column}_sin"] = np.sin(2 * np.pi * original[column] / period)
    original[f"{column}_cos"] = np.cos(2 * np.pi * original[column] / period)
for lag in [1, 2, 3, 6, 12, 24, 48]:
    original[f"price_return_{lag}"] = prices / prices.shift(lag) - 1
ORIGINAL_COLUMNS = list(original.columns)


# -------------------------------
# 7. Labels
# -------------------------------
# Competition training period: the official Signal labels.
# Older history: the same rule the organizer used (a 1% swing low: at least ~1% below the
# highest price since the last lower price, and the price then rises 1% before dropping below
# it). This reproduces 221 of the 222 official training labels (the exception is the very first
# day of the file). Labels only ever use prices up to the cut-off given (never test prices).
def label_with_1pct_rule(price_values, end, threshold=np.log(1.01)):
    p = np.asarray(price_values[:end], dtype=float)
    lp = np.log(p)
    n = len(p)
    left, _ = left_side_of_dip(p)
    next_lower = np.full(n, n)
    stack = []
    for i in range(n):
        while stack and p[i] < p[stack[-1]]:
            next_lower[stack.pop()] = i
        stack.append(i)
    labels = np.full(n, np.nan)
    for i in range(n):
        reached = False
        for k in range(i + 1, min(next_lower[i], n)):
            if lp[k] >= lp[i] + threshold - 1e-12:
                reached = True
                break
        if reached:
            labels[i] = float(left[i] >= threshold - 1e-12 and i > 0 and p[i - 1] > p[i])
        elif next_lower[i] < n:
            labels[i] = 0.0  # price dropped below first -> no signal; otherwise unresolved (NaN)
    out = np.full(len(price_values), np.nan)
    out[:end] = labels
    return out


rule_labels = label_with_1pct_rule(prices.to_numpy(), train_end)
official = train["Signal"].to_numpy()
match = np.mean(rule_labels[n_older:train_end][np.isfinite(rule_labels[n_older:train_end])] ==
                official[np.isfinite(rule_labels[n_older:train_end])])
print(f"1% rule reproduces the official training labels on {match:.2%} of hours")
final_labels = rule_labels.copy()
final_labels[n_older:train_end] = official


# -------------------------------
# 8. The six members
# -------------------------------
def logistic():
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.3, max_iter=3000))


def fit_and_score(fit_rows, score_rows, labels):
    """Train every member on fit_rows (with labels) and score score_rows; non-candidates get 0."""
    fit_rows = fit_rows[np.isfinite(labels[fit_rows])]
    fit_dips = fit_rows[is_candidate[fit_rows]]
    score_dips = is_candidate[score_rows]
    dip_rows = score_rows[score_dips]
    y_dips = labels[fit_dips].astype(int)
    y_all = labels[fit_rows].astype(int)
    scores = {}

    def keep(name, dip_scores):
        full = np.zeros(len(score_rows))
        full[score_dips] = dip_scores
        scores[name] = full

    for name, columns in [("LR trimmed + volume", LR_TRIM_VOLUME_COLUMNS), ("LR volatility", LR_VOL_COLUMNS)]:
        model = logistic().fit(features.loc[fit_dips, columns], y_dips)
        keep(name, model.predict_proba(features.loc[dip_rows, columns])[:, 1])
    keep("Paths (day)", features.loc[dip_rows, "paths_day"].to_numpy())
    keep("Paths (week)", features.loc[dip_rows, "paths_week"].to_numpy())
    keep("(competition6) Paths (day), hourly moves", features.loc[dip_rows, "paths_day_hourly"].to_numpy())
    keep("(competition6) Paths (week), hourly moves", features.loc[dip_rows, "paths_week_hourly"].to_numpy())
    xgb_dips = XGBClassifier(n_estimators=250, max_depth=2, learning_rate=0.03, subsample=0.8, colsample_bytree=0.8,
                             min_child_weight=10, reg_lambda=5, objective="binary:logistic",
                             random_state=42, tree_method="hist", n_jobs=-1)
    xgb_dips.fit(features.loc[fit_dips, XGB_DIP_VOLUME_COLUMNS], y_dips)
    keep("XGB dips + volume", xgb_dips.predict_proba(features.loc[dip_rows, XGB_DIP_VOLUME_COLUMNS])[:, 1])
    xgb_original = XGBClassifier(n_estimators=1000, max_depth=5, learning_rate=0.03, subsample=0.9,
                                 colsample_bytree=0.9, reg_lambda=2, min_child_weight=4,
                                 scale_pos_weight=(y_all == 0).sum() / (y_all == 1).sum(),
                                 objective="binary:logistic", eval_metric="aucpr", random_state=42,
                                 tree_method="hist", n_jobs=-1)
    xgb_original.fit(original.loc[fit_rows, ORIGINAL_COLUMNS], y_all)
    keep("XGB original (returns)", xgb_original.predict_proba(original.loc[dip_rows, ORIGINAL_COLUMNS])[:, 1])
    return scores


MEMBERS = ["LR trimmed + volume", "LR volatility", "Paths (day)", "Paths (week)", "XGB dips + volume",
           "XGB original (returns)"]
COMPETITION6_RECIPE = ["LR trimmed + volume", "LR volatility", "(competition6) Paths (day), hourly moves",
                       "(competition6) Paths (week), hourly moves", "XGB dips + volume", "XGB original (returns)"]

complete = features.notna().all(axis=1).to_numpy() & original.notna().all(axis=1).to_numpy()
all_train_rows = np.flatnonzero(complete[:train_end])
competition_train_rows = all_train_rows[all_train_rows >= n_older]
test_rows = np.arange(train_end, len(timeline))
assert complete[test_rows].all(), "Missing test features"
print(f"Training hours: {len(all_train_rows)} ({len(competition_train_rows)} from the competition) | "
      f"labelled signals: {int(np.nansum(final_labels[all_train_rows]))} | dip candidates: {is_candidate[all_train_rows].sum()}")
print(f"Competition signals that are NOT candidates (should be ~0): "
      f"{int(((official == 1) & ~is_candidate[n_older:train_end]).sum())}")


# -------------------------------
# 9. 5-fold time validation over the competition training period
# -------------------------------
# The competition period is cut into 6 consecutive blocks; fold k trains on everything before
# block k (all older history + earlier competition blocks, minus a 48-hour gap) and is scored
# on block k. Training labels in each fold are re-derived with prices before the gap only.
blocks = np.array_split(competition_train_rows, 6)
fold_scores, fold_labels, fold_dips = [], [], []
for k in range(1, 6):
    valid_rows = blocks[k]
    cutoff = valid_rows[0] - 48
    fold_label_values = label_with_1pct_rule(prices.to_numpy(), cutoff)
    fit_rows = all_train_rows[all_train_rows < cutoff]
    fold_scores.append(fit_and_score(fit_rows, valid_rows, fold_label_values))
    fold_labels.append(official[valid_rows - n_older])
    fold_dips.append(is_candidate[valid_rows])
    print(f"  fold {k} done ({int(official[valid_rows - n_older].sum())} signals)")

# Each member's validation scores on past dips: the yardstick for combining members.
reference = {name: np.sort(np.concatenate([s[name][d] for s, d in zip(fold_scores, fold_dips)]))
             for name in set(MEMBERS) | set(COMPETITION6_RECIPE)}


def combine(scores, is_dip, members=MEMBERS):
    """Turn each member's score into a percentile against that member's own validation
    scores on past dips, then average the members. Uses no test-period information."""
    total = np.zeros(len(is_dip))
    for name in members:
        ref = reference[name]
        total[is_dip] += np.searchsorted(ref, scores[name][is_dip], side="right") / len(ref)
    return total / len(members)


rows = []
for k, (scores, y, is_dip) in enumerate(zip(fold_scores, fold_labels, fold_dips), start=1):
    row = {"fold": k, "signals": int(y.sum())}
    for name, values in scores.items():
        row[name] = average_precision_score(y, values)
    row["competition6 (hourly moves in the paths)"] = average_precision_score(y, combine(scores, is_dip, COMPETITION6_RECIPE))
    row["COMBINED (competition7)"] = average_precision_score(y, combine(scores, is_dip))
    rows.append(row)
validation = pd.DataFrame(rows).set_index("fold")
pd.set_option("display.width", 200)
print("\nValidation Average Precision per fold (higher is better):")
print(validation.round(3).T.to_string())
print(f"\nValidation Average Precision (mean of 5 folds): "
      f"{validation['COMBINED (competition7)'].mean():.4f}  "
      f"(competition6: {validation['competition6 (hourly moves in the paths)'].mean():.4f})")


# -------------------------------
# 10. Final models on all training data (older + competition) -> test predictions
# -------------------------------
final_scores = fit_and_score(all_train_rows, test_rows, final_labels)
probabilities = combine(final_scores, is_candidate[test_rows])
competition6_probabilities = combine(final_scores, is_candidate[test_rows], COMPETITION6_RECIPE)
test_dips = is_candidate[test_rows]
print(f"Test dips ranked like competition6? rank correlation "
      f"{pd.Series(probabilities[test_dips]).corr(pd.Series(competition6_probabilities[test_dips]), method='spearman'):.3f}")

driver_rows = all_train_rows[is_candidate[all_train_rows] & np.isfinite(final_labels[all_train_rows])]
driver_model = logistic().fit(features.loc[driver_rows, LR_TRIM_VOLUME_COLUMNS], final_labels[driver_rows].astype(int))
coefficients = pd.Series(driver_model.named_steps["logisticregression"].coef_[0], index=LR_TRIM_VOLUME_COLUMNS)
print("\nTop drivers (LR trimmed + volume):")
print(coefficients.sort_values(key=abs, ascending=False).round(3).to_string())

submission = pd.DataFrame({"Id": test["Id"].astype(int), "Signal": probabilities})
submission.to_csv("sample_submission.csv", index=False)
print(submission.head())
print(f"Test hours that are dip candidates: {is_candidate[test_rows].sum()} of {len(test_rows)}")
print("Submission written to sample_submission.csv")


Timing check: link with price change into the row -0.76, out of the row +0.02
-> Activity rows cover the hour that ends at each price. Used as they are.
Binance close == competition price on 100% of 40 overlapping hours
Older history: 2022-01-01 00:00:00 -> 2025-09-29 07:00:00 (32816 hours)
Minute candles: 2022-01-01 00:00:00 -> 2026-09-29 07:59:00 (2,494,480 minutes)
Last minute of the hour closes at the competition price: 100.00% of training hours, 99.96% of test hours
   differs at 2026-09-29 07:00:00: competition 84055.59, last minute 83980.97
Minute volumes add up to the competition's hourly volume: 100.00% of hours | last minute == older Binance hourly close: 100.00% of older hours
Hours without enough minutes (left out): 1
1% rule reproduces the official training labels on 99.98% of hours
Training hours: 38775 (6132 from the competition) | labelled signals: 1389 | dip candidates: 5498
Competition signals that are NOT candidates (should be ~0): 1
  fold 1 done (40 signals)
  fold